In [1]:
import sqlite3
import pandas as pd

In [2]:
conn = sqlite3.connect("../database/ecommerce.db")

cursor = conn.cursor()

print("Database Connected Successfully!")

Database Connected Successfully!


In [3]:
customers = pd.read_csv("../data/cleaned/customers_clean.csv")

products = pd.read_csv("../data/cleaned/products_clean.csv")

orders = pd.read_csv("../data/cleaned/orders_clean.csv")

order_items = pd.read_csv("../data/cleaned/order_items_clean.csv")

In [4]:
customers.to_sql(
    "customers",
    conn,
    if_exists="replace",
    index=False
)

products.to_sql(
    "products",
    conn,
    if_exists="replace",
    index=False
)

orders.to_sql(
    "orders",
    conn,
    if_exists="replace",
    index=False
)

order_items.to_sql(
    "order_items",
    conn,
    if_exists="replace",
    index=False
)

print("Tables Created Successfully!")

Tables Created Successfully!


In [5]:
pd.read_sql(
    "SELECT name FROM sqlite_master WHERE type='table';",
    conn
)

,name
0,customers
1,products
2,orders
3,order_items


In [6]:
tables = [
    "customers",
    "products",
    "orders",
    "order_items"
]

for table in tables:

    query = f"SELECT COUNT(*) AS rows FROM {table}"

    print(table)

    print(pd.read_sql(query, conn))

    print("-"*40)

customers
   rows
0   600
----------------------------------------
products
   rows
0   500
----------------------------------------
orders
   rows
0   700
----------------------------------------
order_items
   rows
0  2500
----------------------------------------


In [7]:
import sqlite3

print(sqlite3.sqlite_version)

3.50.4


In [8]:
conn = sqlite3.connect("../database/ecommerce.db")

print("Connected Successfully!")

Connected Successfully!


In [9]:
customers = pd.read_sql("SELECT * FROM customers", conn)

customers.head()

,customer_id,customer_name,email,registration_date,customer_type
0,CUST0001,Aryan Maharaj,udantdewan@example.net,2023-12-18,REGULAR
1,CUST0002,Pahal Balay,chandertejas@example.org,2025-09-04,REGULAR
2,CUST0003,Rushil Saini,saumyamall@example.org,2024-03-16,REGULAR
3,CUST0004,Harini Mall,tanveernayar@example.org,2023-12-03,REGULAR
4,CUST0005,Gunbir Parmer,aishani07@example.net,2024-01-21,PREMIUM


In [10]:
products = pd.read_sql("SELECT * FROM products", conn)

products.head()

,product_id,product_name,category,subcategory,cost_price
0,PROD0001,Dining Table,Home,Furniture,691.21
1,PROD0002,Kurti,Clothing,Women,3740.12
2,PROD0003,Jeans,Clothing,Men,3719.16
3,PROD0004,Lamp,Home,Decor,4904.50
4,PROD0005,Dress,Clothing,Women,1868.32


In [11]:
orders = pd.read_sql("SELECT * FROM orders", conn)

orders.head()

,order_id,customer_id,region,status,order_date
0,ORD00001,CUST0434,South,CANCELLED,2026-03-31 18:36:11
1,ORD00002,CUST0551,South,RETURNED,2026-02-01 12:00:23
2,ORD00003,CUST0171,West,DELIVERED,2025-12-26 06:56:07
3,ORD00004,CUST0144,South,DELIVERED,2025-08-02 16:14:54
4,ORD00005,CUST0462,North,DELIVERED,2025-02-18 15:04:35


In [12]:
order_items = pd.read_sql("SELECT * FROM order_items", conn)

order_items.head()

,order_item_id,order_id,product_id,quantity,unit_price,discount_percent
0,ITEM00001,ORD00140,PROD0177,3,1165.22,0
1,ITEM00002,ORD00504,PROD0485,4,1032.09,49
2,ITEM00003,ORD00441,PROD0409,3,795.76,50
3,ITEM00004,ORD00008,PROD0281,3,5969.29,35
4,ITEM00005,ORD00448,PROD0203,2,5560.17,47


In [13]:
pd.read_sql("""
SELECT name
FROM sqlite_master
WHERE type='table';
""", conn)

,name
0,customers
1,products
2,orders
3,order_items


In [14]:
tables = ["customers","products","orders","order_items"]

for table in tables:

    query = f"SELECT COUNT(*) AS total_rows FROM {table}"

    print(table)

    print(pd.read_sql(query, conn))

    print("-"*50)

customers
   total_rows
0         600
--------------------------------------------------
products
   total_rows
0         500
--------------------------------------------------
orders
   total_rows
0         700
--------------------------------------------------
order_items
   total_rows
0        2500
--------------------------------------------------


In [15]:
query = """

SELECT

    p.category,

    ROUND(

        SUM(

            oi.quantity *

            oi.unit_price *

            (1 - oi.discount_percent/100.0)

        ),

    2) AS total_revenue

FROM order_items oi

JOIN products p

ON oi.product_id = p.product_id

GROUP BY p.category

ORDER BY total_revenue DESC;

"""

pd.read_sql(query,conn)

,category,total_revenue
0,Electronics,4939801.61
1,Home,4907035.01
2,Books,4896691.96
3,Clothing,4810803.70


In [16]:
query = """

SELECT

    c.customer_id,

    c.customer_name,

    ROUND(

        SUM(

            oi.quantity *

            oi.unit_price *

            (1 - oi.discount_percent/100.0)

        ),

    2)

AS total_order_value

FROM customers c

JOIN orders o

ON c.customer_id=o.customer_id

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

    c.customer_id,

    c.customer_name

ORDER BY total_order_value DESC

LIMIT 10;

"""

pd.read_sql(query,conn)

,customer_id,customer_name,total_order_value
0,CUST0047,Nathaniel Chokshi,184436.56
1,CUST0051,Ikbal Saraf,180563.23
2,CUST0536,Adya Ghosh,170810.14
3,CUST0321,Ojas Basu,169130.45
4,CUST0234,Janani Mani,156439.63
5,CUST0494,Sneha Deol,156215.42
6,CUST0041,Yashasvi Suresh,148054.18
7,CUST0270,Osha Prasad,147949.75
8,CUST0500,Udarsh Ramachandran,142166.75
9,CUST0288,Advika Sahni,140659.23


In [17]:
query = """

SELECT

    strftime('%Y-%m',order_date) AS month,

    COUNT(order_id) AS total_orders

FROM orders

GROUP BY month

ORDER BY month DESC

LIMIT 12;

"""

pd.read_sql(query,conn)

,month,total_orders
0,2026-12,4
1,2026-11,5
2,2026-10,9
3,2026-09,9
4,2026-08,7
5,2026-07,6
6,2026-06,16
7,2026-05,22
8,2026-04,19
9,2026-03,32


In [18]:
query = """

SELECT DISTINCT

    c.customer_id,

    c.customer_name

FROM customers c

JOIN orders o

ON c.customer_id=o.customer_id

WHERE NOT EXISTS(

SELECT 1

FROM orders o2

WHERE

o2.customer_id=c.customer_id

AND o2.status='DELIVERED'

);

"""

pd.read_sql(query,conn)

,customer_id,customer_name
0,CUST0002,Pahal Balay
1,CUST0003,Rushil Saini
2,CUST0005,Gunbir Parmer
3,CUST0007,Fariq Kaul
4,CUST0008,Nicholas Prabhakar
...,...,...
271,CUST0587,Nidhi Bora
272,CUST0588,Bishakha Loke
273,CUST0592,Nachiket Chad
274,CUST0593,Jonathan Deep


In [19]:
query = """

SELECT

    p.product_name,

    SUM(

        CASE

            WHEN oi.quantity>0

            THEN oi.quantity

            ELSE 0

        END

    ) AS purchased,

    ABS(

        SUM(

            CASE

                WHEN oi.quantity<0

                THEN oi.quantity

                ELSE 0

            END

        )

    ) AS returned

FROM order_items oi

JOIN products p

ON oi.product_id=p.product_id

GROUP BY p.product_name

HAVING returned>purchased;

"""

pd.read_sql(query,conn)

,product_name,purchased,returned


In [20]:
query = """

SELECT

    p.category,

    ROUND(

        100.0 *

        ABS(

            SUM(

                CASE

                    WHEN oi.quantity<0

                    THEN oi.quantity

                    ELSE 0

                END

            )

        )

        /

        SUM(ABS(oi.quantity))

    ,2)

AS return_rate

FROM order_items oi

JOIN products p

ON oi.product_id=p.product_id

GROUP BY p.category;

"""

pd.read_sql(query,conn)

,category,return_rate
0,Books,1.99
1,Clothing,4.21
2,Electronics,3.26
3,Home,3.37


In [21]:
conn.close()

print("Database Connection Closed.")

Database Connection Closed.


In [22]:
conn.close()

print("Database Connection Closed.")

Database Connection Closed.


In [24]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../database/ecommerce.db")

In [25]:
query = """

SELECT

    region,

    DATE(order_date) AS order_date,

    ROUND(
        SUM(
            oi.quantity *
            oi.unit_price *
            (1 - oi.discount_percent/100.0)
        ),
    2) AS daily_revenue,

    ROUND(

        SUM(

            SUM(
                oi.quantity *
                oi.unit_price *
                (1 - oi.discount_percent/100.0)
            )

        ) OVER(

            PARTITION BY region

            ORDER BY DATE(order_date)

        ),

    2)

AS running_total

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

region,

DATE(order_date)

ORDER BY

region,

DATE(order_date);

"""

pd.read_sql(query,conn)

,region,order_date,daily_revenue,running_total
0,East,2024-01-10,29934.48,29934.48
1,East,2024-02-11,1680.70,31615.18
2,East,2024-05-08,11122.46,42737.63
3,East,2024-05-09,39000.92,81738.55
4,East,2024-05-12,-9375.33,72363.22
...,...,...,...,...
615,West,2026-09-02,42193.97,4357915.01
616,West,2026-09-07,5502.18,4363417.19
617,West,2026-11-04,15848.23,4379265.41
618,West,2026-11-07,14167.55,4393432.96


In [26]:
query = """

SELECT

    category,

    product_name,

    total_revenue,

    DENSE_RANK()

OVER(

PARTITION BY category

ORDER BY total_revenue DESC

)

AS rank_in_category

FROM(

SELECT

p.category,

p.product_name,

ROUND(

SUM(

oi.quantity*

oi.unit_price*

(1-oi.discount_percent/100.0)

),2)

AS total_revenue

FROM products p

JOIN order_items oi

ON p.product_id=oi.product_id

GROUP BY

p.category,

p.product_name

);

"""

pd.read_sql(query,conn)

,category,product_name,total_revenue,rank_in_category
0,Books,Mystery Novel,938948.81,1
1,Books,Manga Volume,623491.99,2
2,Books,Thriller Novel,582764.65,3
3,Books,Data Structures,551655.63,4
4,Books,Fantasy Book,546684.86,5
5,Books,Marvel Comic,460470.23,6
6,Books,Python Programming,326999.49,7
7,Books,Operating Systems,295900.09,8
8,Books,Dbms,287225.23,9
9,Books,Batman Comic,282550.97,10


In [27]:
query = """

SELECT

customer_id,

order_date,

LAG(order_date)

OVER(

PARTITION BY customer_id

ORDER BY order_date

)

AS previous_order_date,

ROUND(

julianday(order_date)-

julianday(

LAG(order_date)

OVER(

PARTITION BY customer_id

ORDER BY order_date

)

),

2

)

AS days_gap

FROM orders

WHERE customer_id!='UNKNOWN';

"""

pd.read_sql(query,conn)

,customer_id,order_date,previous_order_date,days_gap
0,CUST0002,2026-01-02 09:17:08,NaN,NaN
1,CUST0003,2024-10-17 15:12:53,NaN,NaN
2,CUST0003,2024-12-11 08:56:17,2024-10-17 15:12:53,54.74
3,CUST0005,2025-08-25 05:14:43,NaN,NaN
4,CUST0005,2025-09-28 06:24:04,2025-08-25 05:14:43,34.05
...,...,...,...,...
660,CUST0597,2024-07-23 04:19:20,NaN,NaN
661,CUST0597,2026-02-22 11:32:34,2024-07-23 04:19:20,579.30
662,CUST0598,2024-05-09 04:26:42,NaN,NaN
663,CUST0600,2024-08-13 02:08:41,NaN,NaN


In [28]:
query = """

WITH monthly_revenue AS(

SELECT

o.customer_id,

strftime('%Y-%m',o.order_date) AS month,

SUM(

oi.quantity*

oi.unit_price*

(1-oi.discount_percent/100.0)

)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

WHERE o.customer_id!='UNKNOWN'

GROUP BY

o.customer_id,

month

),

customer_segment AS(

SELECT

customer_id,

month,

CASE

WHEN revenue>10000 THEN 'High'

WHEN revenue>=5000 THEN 'Medium'

ELSE 'Low'

END

AS segment

FROM monthly_revenue

)

SELECT

month,

segment,

COUNT(*) AS customers

FROM customer_segment

GROUP BY

month,

segment

ORDER BY

month,

segment;

"""

pd.read_sql(query,conn)

,month,segment,customers
0,2024-01,High,1
1,2024-01,Low,1
2,2024-02,High,3
3,2024-02,Low,1
4,2024-03,High,2
...,...,...,...
82,2026-10,High,7
83,2026-10,Low,1
84,2026-11,High,4
85,2026-11,Medium,1


In [29]:
query = """

WITH customer_value AS(

SELECT

o.customer_id,

ROUND(

SUM(

oi.quantity*

oi.unit_price*

(1-oi.discount_percent/100.0)

),2)

AS total_value

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

WHERE o.customer_id!='UNKNOWN'

GROUP BY o.customer_id

)

SELECT

customer_id,

total_value,

NTILE(4)

OVER(

ORDER BY total_value DESC

)

AS quartile,

CASE

NTILE(4)

OVER(

ORDER BY total_value DESC

)

WHEN 1 THEN 'Platinum'

WHEN 2 THEN 'Gold'

WHEN 3 THEN 'Silver'

ELSE 'Bronze'

END

AS quartile_label

FROM customer_value;

"""

pd.read_sql(query,conn)

,customer_id,total_value,quartile,quartile_label
0,CUST0047,184436.56,1,Platinum
1,CUST0051,180563.23,1,Platinum
2,CUST0536,170810.14,1,Platinum
3,CUST0321,169130.45,1,Platinum
4,CUST0234,156439.63,1,Platinum
...,...,...,...,...
383,CUST0157,761.38,4,Bronze
384,CUST0458,700.53,4,Bronze
385,CUST0173,165.45,4,Bronze
386,CUST0263,-1710.94,4,Bronze


In [30]:
conn.close()

print("Queries 7-11 Completed Successfully!")

Queries 7-11 Completed Successfully!


In [31]:
import sqlite3
import pandas as pd

conn = sqlite3.connect("../database/ecommerce.db")

In [32]:
query = """

WITH yearly_revenue AS (

SELECT

strftime('%Y',o.order_date) AS year,

strftime('%m',o.order_date) AS month,

ROUND(

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100.0)

),2)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

GROUP BY

year,

month

)

SELECT

y1.year,

y1.month,

y1.revenue,

y2.revenue AS prev_year_revenue,

ROUND(

((y1.revenue-y2.revenue)*100.0)/y2.revenue,

2

)

AS yoy_growth_percent

FROM yearly_revenue y1

LEFT JOIN yearly_revenue y2

ON

y1.month=y2.month

AND CAST(y1.year AS INTEGER)=CAST(y2.year AS INTEGER)+1

ORDER BY

y1.year,

y1.month;

"""

pd.read_sql(query,conn)

,year,month,revenue,prev_year_revenue,yoy_growth_percent
0,2024,01,31633.43,NaN,NaN
1,2024,02,172464.11,NaN,NaN
2,2024,03,39687.82,NaN,NaN
3,2024,04,116985.57,NaN,NaN
4,2024,05,150730.43,NaN,NaN
5,2024,06,206591.72,NaN,NaN
6,2024,07,799651.67,NaN,NaN
7,2024,08,787749.06,NaN,NaN
8,2024,09,746296.45,NaN,NaN
9,2024,10,444214.82,NaN,NaN


In [33]:
query = """

WITH customer_orders AS (

SELECT

o.customer_id,

o.order_date,

p.category,

ROW_NUMBER()

OVER(

PARTITION BY o.customer_id

ORDER BY o.order_date

)

AS first_order,

ROW_NUMBER()

OVER(

PARTITION BY o.customer_id

ORDER BY o.order_date DESC

)

AS last_order

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

JOIN products p

ON oi.product_id=p.product_id

WHERE o.customer_id!='UNKNOWN'

)

SELECT

f.customer_id,

f.category AS first_category,

l.category AS latest_category,

CASE

WHEN f.category=l.category

THEN 'No'

ELSE 'Yes'

END

AS category_shift

FROM customer_orders f

JOIN customer_orders l

ON f.customer_id=l.customer_id

WHERE

f.first_order=1

AND l.last_order=1;

"""

pd.read_sql(query,conn)

,customer_id,first_category,latest_category,category_shift
0,CUST0002,Books,Books,No
1,CUST0003,Clothing,Electronics,Yes
2,CUST0005,Home,Electronics,Yes
3,CUST0007,Home,Books,Yes
4,CUST0008,Home,Home,No
...,...,...,...,...
383,CUST0593,Electronics,Electronics,No
384,CUST0595,Electronics,Electronics,No
385,CUST0597,Books,Clothing,Yes
386,CUST0598,Home,Home,No


In [34]:
query = """

WITH customer_revenue AS(

SELECT

o.customer_id,

ROUND(

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100.0)

),2)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

WHERE o.customer_id!='UNKNOWN'

GROUP BY o.customer_id

)

SELECT

customer_id,

revenue,

SUM(revenue)

OVER(

ORDER BY revenue DESC

)

AS cumulative_revenue,

ROUND(

100.0*

SUM(revenue)

OVER(

ORDER BY revenue DESC

)

/

SUM(revenue)

OVER(),

2

)

AS cumulative_percent

FROM customer_revenue

ORDER BY revenue DESC;

"""

pd.read_sql(query,conn)

,customer_id,revenue,cumulative_revenue,cumulative_percent
0,CUST0047,184436.56,184436.56,0.99
1,CUST0051,180563.23,364999.79,1.96
2,CUST0536,170810.14,535809.93,2.87
3,CUST0321,169130.45,704940.38,3.78
4,CUST0234,156439.63,861380.01,4.62
...,...,...,...,...
383,CUST0157,761.38,18641294.09,100.02
384,CUST0458,700.53,18641994.62,100.02
385,CUST0173,165.45,18642160.07,100.02
386,CUST0263,-1710.94,18640449.13,100.01


In [35]:
query = """

WITH first_purchase AS(

SELECT

customer_id,

MIN(DATE(order_date))

AS cohort_date

FROM orders

WHERE customer_id!='UNKNOWN'

GROUP BY customer_id

),

customer_orders AS(

SELECT

o.customer_id,

DATE(o.order_date)

AS order_date,

strftime('%Y-%m',f.cohort_date)

AS cohort,

(

CAST(strftime('%Y',o.order_date) AS INTEGER)

-

CAST(strftime('%Y',f.cohort_date) AS INTEGER)

)*12+

(

CAST(strftime('%m',o.order_date) AS INTEGER)

-

CAST(strftime('%m',f.cohort_date) AS INTEGER)

)

AS month_number

FROM orders o

JOIN first_purchase f

ON o.customer_id=f.customer_id

)

SELECT

cohort,

COUNT(

DISTINCT CASE WHEN month_number=0 THEN customer_id END

)

AS month0,

COUNT(

DISTINCT CASE WHEN month_number=1 THEN customer_id END

)

AS month1,

COUNT(

DISTINCT CASE WHEN month_number=2 THEN customer_id END

)

AS month2,

COUNT(

DISTINCT CASE WHEN month_number=3 THEN customer_id END

)

AS month3

FROM customer_orders

GROUP BY cohort

ORDER BY cohort;

"""

pd.read_sql(query,conn)

,cohort,month0,month1,month2,month3
0,2024-01,2,0,0,0
1,2024-02,5,0,0,0
2,2024-03,4,0,0,0
3,2024-04,4,0,0,0
4,2024-05,6,0,0,0
5,2024-06,9,0,1,1
6,2024-07,23,0,4,1
7,2024-08,25,0,1,1
8,2024-09,15,0,0,1
9,2024-10,15,1,1,1


In [36]:
query = """

WITH customer_sales AS(

SELECT

o.customer_id,

ROUND(

SUM(

oi.quantity*
oi.unit_price*
(1-oi.discount_percent/100.0)

),2)

AS revenue

FROM orders o

JOIN order_items oi

ON o.order_id=oi.order_id

WHERE o.customer_id!='UNKNOWN'

GROUP BY o.customer_id

)

SELECT

customer_id,

revenue,

LAG(revenue)

OVER(

ORDER BY revenue DESC

)

AS previous_customer_revenue

FROM customer_sales;

"""

pd.read_sql(query,conn)

,customer_id,revenue,previous_customer_revenue
0,CUST0047,184436.56,NaN
1,CUST0051,180563.23,184436.56
2,CUST0536,170810.14,180563.23
3,CUST0321,169130.45,170810.14
4,CUST0234,156439.63,169130.45
...,...,...,...
383,CUST0157,761.38,1748.37
384,CUST0458,700.53,761.38
385,CUST0173,165.45,700.53
386,CUST0263,-1710.94,165.45


In [37]:
conn.close()

print("All 16 SQL Queries Executed Successfully!")

All 16 SQL Queries Executed Successfully!
